In [1]:
import os.path
import random
from PIL import Image, ImageDraw, ImageFont
import cv2
import numpy as np
import torch
import torch.nn as nn
from torch import optim
from torch.utils.data import Dataset, DataLoader

### Генерация текста

In [2]:

class StringGenerator:

    @staticmethod
    def text_generator(lang):
        string = ''
        alphabet_rus_small = 'абвгдеёжзийклмопрстфхцшщьыъэюя'
        alphabet_rus_big = 'АБВГДЕЁЖЗИЙКЛМНОПРСТУФХЦЧШЩЪЫЬЭЮЯ'
        alphabet_eng_small = 'abcdefghijklmnopqrstuvwxyz'
        alphabet_eng_big = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ'
        choice = random.randint(1, 3)
        if lang == 'rus':
            if choice == 1:
                for i in range(3):
                    string += random.choice(alphabet_rus_small)
            elif choice == 2:
                for i in range(3):
                    string += random.choice(alphabet_rus_big)
            elif choice == 3:
                string = random.choice(alphabet_rus_big) + random.choice(alphabet_rus_small) + random.choice(alphabet_rus_small)
        elif lang == 'eng':
            if choice == 1:
                for i in range(3):
                    string += random.choice(alphabet_eng_small)
            elif choice == 2:
                for i in range(3):
                    string += random.choice(alphabet_eng_big)
            elif choice == 3:
                string = random.choice(alphabet_eng_big) + random.choice(alphabet_eng_small) + random.choice(alphabet_eng_small)
        return string


# print(StringGenerator.text_generator('eng'))

# Текст текст ТЕКСТ


### Генерация разных пар картинок с текстом

In [23]:



class FontGenerator:
    def __init__(self):
        self.fonts = [os.path.join('fonts', name) for name in [
            'arial.ttf',  # Arial
            'calibri.ttf',  # Calibri
            'times.ttf',  # Times New Roman
            'impact.ttf',  # Impact
            'ariblk.ttf',  # Arial Black
            'cour.ttf',  # Courier New
            'consola.ttf',  # Consolas
            'CascadiaMono.ttf',  # Cascadis Mono
            'verdana.ttf',  # Verdana
            'tahoma.ttf',  # Tahoma
            'lucon.ttf',  # Lucida Console
            'GARA.ttf',  # Garamond
            'BKANT.ttf',  # Book Antiqua
            'cambria.ttc',  # Cambria
            'constan.ttf',  # Constantia
            'segoesc.ttf',  # Segoe Script
            'comic.ttf', # Comic Sans MS
            'MTCORSVA.ttf'  # Monotype Corsiva
        ]]

        self.image_size = (120, 60)
        self.font_size = 40
        self.intervals = [
            (-10, 10),  # отклонение по ширине
            (-20, 10)  # отклонение по высоте
        ]

    def random_position_with_constraints(self):
        # разделяем интервалы для ширины (x) и высоты (y)
        x_interval, y_interval = self.intervals

        # генерация случайной позиции по ширине
        x = random.randint(x_interval[0], x_interval[1])

        # генерация случайной позиции по высоте
        y = random.randint(y_interval[0], y_interval[1])

        return (x, y)

    def draw_font(self, text, font_path, image_size, font_size):
        image = Image.new('RGB', image_size, 'white')  # изображение с белым фоном
        draw = ImageDraw.Draw(image)

        font = ImageFont.truetype(font_path, font_size)

        position = self.random_position_with_constraints()

        draw.text(position, text, fill='black', font=font)

        return image

    def generate_images(self, index, answer, style=False, same_text=False):
        lang = random.choice(['rus', 'eng'])
        # одинаковый шрифт
        if style:
            font_path = random.choice(self.fonts)
            font_name = os.path.basename(font_path).split('.')[0]
            images = []
            for i in range(2):
                text = StringGenerator.text_generator(lang)
                images.append(self.draw_font(text, font_path, self.image_size, self.font_size))
        # одинаковый текст
        elif same_text:
            text = StringGenerator.text_generator(lang)
            images = []
            for i in range(2):
                font_path = random.choice(self.fonts)
                font_name = os.path.basename(font_path).split('.')[0]
                images.append(self.draw_font(text, font_path, self.image_size, self.font_size))
        # все разное
        else:
            images = []
            for i in range(2):
                font_path = random.choice(self.fonts)
                font_name = os.path.basename(font_path).split('.')[0]
                text = StringGenerator.text_generator(lang)
                images.append(self.draw_font(text, font_path, self.image_size, self.font_size))
        final_image = Image.new('RGB', (images[0].width + images[1].width, images[1].height))
        final_image.paste(images[0], (0, 0))
        final_image.paste(images[1], (images[0].width, 0))
        final_image.save(os.path.join('dataset', f'image_{index}.png'))

# Раскомментировать чтобы сгенерировать 10000 картинок

# if __name__ == '__main__':
#     font_generator = FontGenerator()
#     os.mkdir('dataset')
#     os.mkdir(os.path.join('dataset', '0'))
#     os.mkdir(os.path.join('dataset', '1'))
    
#     for i in range(5000):
#         if i < 2500:
#             font_generator.generate_images(i, answer=0)
#         else:
#             font_generator.generate_images(i, answer=0, same_text=True)
#     for i in range(5000):
#         font_generator.generate_images(i, answer=1, style=True)



In [4]:
class CharImageDataset(Dataset):
    def __init__(self, img_dir, transform=None, target_transform=None):
        self.img_dir = img_dir
        self.labels = ['0', '1']
        self.counts = [len(os.listdir(os.path.join(self.img_dir, label))) for label in self.labels]
        self.count = sum(self.counts)
        self.transform = transform
        self.target_transform = target_transform

    def __len__(self):
        return self.count

    def __getitem__(self, idx):
        label, i = self.__get_label_and_i_from_idx(idx)
        img_path = os.path.join(self.img_dir, label, f"image_{i}.png")
        image = Image.open(img_path)
        image_array = np.array(image)

        if self.transform:
            image_array = self.transform(image_array)
        if self.target_transform:
            label = self.target_transform(label)
        return image_array, label

    def __get_label_and_i_from_idx(self, idx):
        k = 0
        while (idx - self.counts[k]) >= 0:
            idx -= self.counts[k]
            k += 1
        return self.labels[k], idx

def label_to_vec(text):
    return torch.Tensor([float(text[0])])

def image_to_gray(image):
    grayscale_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY) / 255.0
    return torch.Tensor(grayscale_image).unsqueeze(0)  # размерность канала

### Модель

In [5]:
class CharCNNClassifier(nn.Module):
    def __init__(self):
        super(CharCNNClassifier, self).__init__()

        # Сверточные слои
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, stride=1, padding=1)

        # Максимальный пулинг
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Полносвязные слои
        self.fc1 = nn.Linear(32 * 15 * 60, 128)  # Обновлено
        self.fc2 = nn.Linear(128, 1)  # Выходной вектор длины 1

        # Функция активации
        self.relu = nn.ReLU()

    def forward(self, x):
        # Применяем свертки и пулинг
        x = self.pool(self.relu(self.conv1(x)))  # (batch_size, 16, 30, 120)
        x = self.pool(self.relu(self.conv2(x)))  # (batch_size, 32, 15, 60)

        # Выравниваем тензор для полносвязного слоя
        x = x.view(x.size(0), -1)  # (batch_size, 32 * 15 * 60)

        # Применяем полносвязные слои
        x = self.relu(self.fc1(x))  # (batch_size, 128)
        x = self.fc2(x)  # (batch_size, 1)

        return x




### Обучение модели

In [6]:
dataset = CharImageDataset("dataset/", transform=image_to_gray, target_transform=label_to_vec)

num_epochs = 20

# model = CharCNNClassifier()
#
# example_batch_x = torch.stack([dataset[i][0] for i in range(2)])  # (2, 1, 60, 240)
# example_batch_y = torch.stack([dataset[i][1] for i in range(2)])  # (2, 1)
#
# # проверка входных данных
# print("example_batch_x shape:", example_batch_x.shape)  # (2, 1, 60, 240)
# print("example_batch_y shape:", example_batch_y.shape)  # (2, 1)
#
# output = model(example_batch_x)
#
# # проверка выходных данных
# print("Output shape:", output.shape)  # (2, 1)
# print("Output:", output)
# print("Labels:", example_batch_y)


model = CharCNNClassifier()
criterion = nn.BCEWithLogitsLoss() # функция потерь
optimizer = optim.Adam(model.parameters(), lr=0.001)
train = DataLoader(dataset, batch_size=128, shuffle=True)
for epoch in range(num_epochs):
    model.train()  # Переводим модель в режим обучения
    running_loss = 0.0

    for i, (inputs, targets) in enumerate(train):
        # Обнуляем градиенты
        optimizer.zero_grad()

        # Прямой проход
        outputs = model(inputs)

        # Вычисление потерь
        loss = criterion(outputs, targets)

        # Обратное распространение и обновление весов
        loss.backward()
        optimizer.step()

        # Суммируем потери для вывода
        running_loss += loss.item()

        # Выводим статистику каждые 10 батчей
        if i % 10 == 9:
            print(f"Epoch [{epoch + 1}/{num_epochs}], Batch [{i + 1}/{len(train)}], Loss: {running_loss / 10:.4f}")
            running_loss = 0.0

Epoch [1/20], Batch [10/79], Loss: 0.9715
Epoch [1/20], Batch [20/79], Loss: 0.6963
Epoch [1/20], Batch [30/79], Loss: 0.6954
Epoch [1/20], Batch [40/79], Loss: 0.6954
Epoch [1/20], Batch [50/79], Loss: 0.6943
Epoch [1/20], Batch [60/79], Loss: 0.6936
Epoch [1/20], Batch [70/79], Loss: 0.6945
Epoch [2/20], Batch [10/79], Loss: 0.6917
Epoch [2/20], Batch [20/79], Loss: 0.6928
Epoch [2/20], Batch [30/79], Loss: 0.6922
Epoch [2/20], Batch [40/79], Loss: 0.6919
Epoch [2/20], Batch [50/79], Loss: 0.6922
Epoch [2/20], Batch [60/79], Loss: 0.6900
Epoch [2/20], Batch [70/79], Loss: 0.6912
Epoch [3/20], Batch [10/79], Loss: 0.6830
Epoch [3/20], Batch [20/79], Loss: 0.6814
Epoch [3/20], Batch [30/79], Loss: 0.6799
Epoch [3/20], Batch [40/79], Loss: 0.6747
Epoch [3/20], Batch [50/79], Loss: 0.6561
Epoch [3/20], Batch [60/79], Loss: 0.6434
Epoch [3/20], Batch [70/79], Loss: 0.5984
Epoch [4/20], Batch [10/79], Loss: 0.5313
Epoch [4/20], Batch [20/79], Loss: 0.4960
Epoch [4/20], Batch [30/79], Loss:

### Проверка работы модели

In [7]:
def classifier(model, char):
    gray_image = image_to_gray(char)
    data = gray_image.unsqueeze(0)
    rez = model(data)
# Применяем сигмоидную функцию для получения вероятности
    probability = torch.sigmoid(rez)
    return torch.mean(probability, 0)  # Возвращаем среднее значение вероятности

def interpretation_class(v):
    if v < 0.5:
        print("Шрифты разные")
    else:
        print("Шрифты одинаковые")

In [15]:
# Картинка 1 (разный текст и одинаковый шрифт)
FontGenerator().generate_images(1, answer=1, style=True)

image_path = 'dataset/image_1.png'
image = Image.open(image_path)
image.show()
image_array = np.array(image)

# проверка размерности
# print("image_array shape:", image_array.shape)  # (60, 240, 3)

v = classifier(model, image_array)
print(v)
interpretation_class(v)

tensor([1.], grad_fn=<MeanBackward1>)
Шрифты одинаковые


In [16]:
# Картинка 2 (разный текст и разный шрифт)
FontGenerator().generate_images(2, answer=0)

image_path = 'dataset/image_2.png'
image = Image.open(image_path)
image.show()
image_array = np.array(image)

# проверка размерности
# print("image_array shape:", image_array.shape)  # (60, 240, 3)

v = classifier(model, image_array)
print(v)
interpretation_class(v)

tensor([1.7176e-09], grad_fn=<MeanBackward1>)
Шрифты разные


In [25]:
# Картинка 3 (одинаковый текст и разный шрифт)

FontGenerator().generate_images(3, answer=0, same_text=True)

image_path = 'dataset/image_3.png'
image = Image.open(image_path)
image.show()
image_array = np.array(image)

# проверка размерности
# print("image_array shape:", image_array.shape)  # (60, 240, 3)

v = classifier(model, image_array)
print(v)
interpretation_class(v)

tensor([1.7591e-08], grad_fn=<MeanBackward1>)
Шрифты разные
